In [58]:
# %%
# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

#v1 - rmse from optimization
#v2 - rmse of dTi from val
#v3 - remse from optimization (updated weights
# --- paths (edit if needed) ---
PARAMS_CSV_1R_agg = "../trained_params_lin_regres_agg_night_v2.csv"
PARAMS_CSV_1R_dis = "../trained_params_lin_regres_disag_night_v2.csv"

PARAMS_CSV_2R_agg = "../trained_params_lin_regres_agg_2R_v1.csv"
PARAMS_CSV_2R_dis = "../trained_params_lin_regres_disag_2R_v1.csv"   # <-- add
# .csv

def read_params_wide(path):
    df = pd.read_csv(path)

    param_col = df.columns[0]
    df = df.set_index(param_col).T
    df.index.name = "Property_ID"

    # numeric params (ADD R_m)
    for c in ["C", "R_a", "R_m", "w_s", "w", "w_n"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in ["train_start", "train_end", "val_start", "val_end"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    out = df.reset_index()
    out["Property_ID"] = out["Property_ID"].astype(str)
    return out

df_params_agg_1R = read_params_wide(PARAMS_CSV_1R_agg)
df_params_dis_1R = read_params_wide(PARAMS_CSV_1R_dis)
df_params_agg_2R = read_params_wide(PARAMS_CSV_2R_agg)  # <-- fix (was wrong in your snippet)
df_params_dis_2R = read_params_wide(PARAMS_CSV_2R_dis)  # <-- fix


Q_STREAMS_PARQUET = "../retrieved_weather_data/q_streams_30min.parquet"
#DETACHED_PARQUET  = "training_data/data_detached_with_weather.parquet"
# (or your merged file) e.g.
DETACHED_PARQUET = "../retrieved_weather_data/merged_data_final_homes.parquet"

# validation settings
WEEK_DAYS = 7
DELTA_T_HOURS = 0.5          # 30-min
MAX_WEEK_SEARCH_DAYS = 365   # how far forward we’ll search after training end
MAX_SHIFT_TRIES = MAX_WEEK_SEARCH_DAYS  # shift forward by 1 day each try
INTERP_LIMIT_STEPS = 4       # short-gap interpolation limit (4*30min = 2 hours)



In [59]:
# %%
def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}")

def _has_col(df, name):
    return name in df.columns


In [60]:
# %%
# %%
def read_q_exact_30min(df_train_q_exact: pd.DataFrame,
                       id_use,
                       t_start: pd.Timestamp,
                       t_end: pd.Timestamp) -> pd.DataFrame:
    """
    df_train_q_exact: index = Timestamp (30-min)
      columns = MultiIndex [Property_ID, variable]
        variable in {"Q_hp_total","Q_immersion","Q_dhw","Q_hp_sc","Q_total"}

    Returns: single-home df with flat columns + Q_spc_exact and Q_dhw_exact (if possible).
    """
    df_win = df_train_q_exact.loc[t_start:t_end]

    if not isinstance(df_win.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns [Property_ID, variable].")

    if id_use not in df_win.columns.get_level_values(0):
        raise KeyError(f"id_use={id_use} not found in df_train_q_exact columns.")

    df_id = df_win.loc[:, (id_use, slice(None))].copy()
    df_id.columns = df_id.columns.droplevel(0)

    if "Q_dhw" in df_id.columns:
        df_id["Q_dhw_exact"] = df_id["Q_dhw"]

    if "Q_total" in df_id.columns and "Q_dhw" in df_id.columns:
        df_id["Q_spc_exact"] = df_id["Q_total"] - df_id["Q_dhw"]

    return df_id



In [61]:
# %%
def build_df_home(df_detached, id_use):
    df_home = df_detached[df_detached["Property_ID"] == id_use].copy()
    df_home = df_home.sort_index()

    # Engineer columns used by your model (from your pasted script)
    # If any base columns are missing in your parquet, adjust here.
    df_home["Heat_Pump_Energy_Output_Diff"] = df_home["Heat_Pump_Energy_Output"].diff()
    df_home["Internal_Temperature_Diff"] = df_home["Internal_Air_Temperature"].diff()
    df_home["Internal_Ambient_Temperature_Diff"] = (
        df_home["temp"] - df_home["Internal_Air_Temperature"]
    )

    # Interpolate short gaps on numeric columns
    num_cols = df_home.select_dtypes(include=["number"]).columns
    df_home[num_cols] = df_home[num_cols].interpolate(
        method="time", limit=INTERP_LIMIT_STEPS, limit_direction="both"
    )

    # Drop remaining NaNs required for the model
    must = [
        "Internal_Air_Temperature",
        "temp",
        "Internal_Ambient_Temperature_Diff",
        "Internal_Temperature_Diff",
        "solarradiation",
    ]
    # Heat_Pump_Energy_Output_Diff not strictly needed for validation, but keep if present
    for c in must:
        if c not in df_home.columns:
            raise KeyError(f"Missing required column in df_home: {c}")

    df_home = df_home.dropna(subset=must)
    return df_home


In [62]:
# %%
def next_midnight(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    # go to next day midnight if not already exactly midnight
    if ts.hour == 0 and ts.minute == 0 and ts.second == 0:
        return ts
    return (ts.normalize() + pd.Timedelta(days=1))

def find_validation_week(
    id_use,
    df_home,
    df_q_exact_all,
    train_end: pd.Timestamp,
    week_days=WEEK_DAYS,
    max_shift_tries=MAX_SHIFT_TRIES,
):
    """
    Choose a 7-day window strictly AFTER train_end that has:
      - enough df_home points
      - full Q_spc_exact alignment (no NaNs after reindex)
    Strategy:
      start at next midnight after train_end, then shift forward 1 day until valid.
    """
    t0 = next_midnight(train_end + pd.Timedelta(minutes=30))  # ensure strictly outside
    week = pd.Timedelta(days=week_days) - pd.Timedelta(minutes=30)
    need_pts = int((24 * week_days) / DELTA_T_HOURS)  # e.g., 336 for 7 days at 30-min

    for shift in range(max_shift_tries):
        start = t0 + pd.Timedelta(days=shift)
        end = start + week

        df_week = df_home.loc[start:end].copy()
        if len(df_week) < need_pts:
            continue

        # exact q streams for same window
        try:
            df_q_week = read_q_exact_30min(df_q_exact_all, id_use, start, end)
        except Exception:
            continue

        if "Q_spc_exact" not in df_q_week.columns:
            continue

        # Align q to df_week index
        df_q_aligned = df_q_week.reindex(df_week.index)

        if df_q_aligned["Q_spc_exact"].isna().any():
            continue

        return start, end, df_week, df_q_aligned

    raise RuntimeError(
        f"Could not find a valid {week_days}-day validation week for home {id_use} "
        f"after train_end={train_end} within {max_shift_tries} day-shifts."
    )


In [63]:
def find_validation_month(
    id_use,
    df_home,
    df_q_exact_all,
    train_end: pd.Timestamp,
    month_days: int = 30,
    max_shift_tries: int = MAX_SHIFT_TRIES,
):
    """
    Choose a ~month window strictly AFTER train_end that has:
      - enough df_home points
      - full Q_spc_exact alignment (no NaNs after reindex)
    Strategy:
      start at next midnight after train_end, then shift forward 1 day until valid.

    Returns:
      start, end, df_month, df_q_aligned
    """
    # ensure strictly outside training window
    t0 = next_midnight(train_end + pd.Timedelta(minutes=30))

    horizon = pd.Timedelta(days=month_days) - pd.Timedelta(minutes=30)
    need_pts = int((24 * month_days) / DELTA_T_HOURS)  # e.g., 30 days * 48 = 1440

    for shift in range(max_shift_tries):
        start = t0 + pd.Timedelta(days=shift)
        end = start + horizon

        df_month = df_home.loc[start:end].copy()
        if len(df_month) < need_pts:
            continue

        # exact q streams for same window
        try:
            df_q_month = read_q_exact_30min(df_q_exact_all, id_use, start, end)
        except Exception:
            continue



        if "Q_spc_exact" not in df_q_month.columns:
            continue

        # Align q to df_month timeline (strict)
        df_q_aligned = df_q_month.reindex(df_month.index)

        # must be fully observed
        if df_q_aligned["Q_spc_exact"].isna().any():
            continue

        return start, end, df_month, df_q_aligned

    raise RuntimeError(
        f"Could not find a valid {month_days}-day validation month for home {id_use} "
        f"after train_end={train_end} within {max_shift_tries} day-shifts."
    )


In [64]:
# %%
def rmse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[mask] - b[mask]) ** 2)))

def simulate_week(params_row: pd.Series,
                  df_week: pd.DataFrame,
                  df_q_week_aligned: pd.DataFrame,
                  delta_t_hours=0.5):
    """
    Supports:
      - 1R1C: C dTi/dt = (Ta-Ti)/Ra + q + w_s*q_s + w + w_n*idx
      - 2R1C: C dTi/dt = (Ta-Ti)/Ra + (Tm-Ti)/Rm + q + w_s*q_s + w + w_n*idx
        with Tm = average Ti over past 24h (strict), using rolling mean + shift(1).

    Returns same metrics as before.
    """
    C   = float(params_row["C"])
    R_a = float(params_row["R_a"])
    w_s = float(params_row.get("w_s", 0.0))
    w_const = float(params_row.get("w", 0.0))
    w_n     = float(params_row.get("w_n", 0.0))

    use_2R = ("R_m" in params_row.index) and pd.notna(params_row["R_m"])
    R_m = float(params_row["R_m"]) if use_2R else None

    # --- if 2R, build T_m (past 24h mean of Ti) and drop first 24h ---
    if use_2R:
        steps_24h = int(round(24 / delta_t_hours))  # 24h / 0.5h = 48
        Ti_series = df_week["Internal_Air_Temperature"].astype(float)

        T_m_series = (
            Ti_series.rolling(window=steps_24h, min_periods=steps_24h).mean().shift(1)
        )

        # Keep only rows where T_m is defined
        keep = T_m_series.notna()
        df_week = df_week.loc[keep].copy()
        df_q_week_aligned = df_q_week_aligned.reindex(df_week.index)

        # If q becomes NaN after reindexing, bail
        if df_q_week_aligned["Q_spc_exact"].isna().any():
            raise RuntimeError("2R1C validation window has insufficient Q_spc_exact after T_m filtering.")

        df_week["T_m_24h"] = T_m_series.loc[df_week.index]

    L = len(df_week) - 1
    if L <= 0:
        raise RuntimeError("Week slice too short after filtering.")

    # exogenous vectors (length L)
    delta_T_a = df_week["Internal_Ambient_Temperature_Diff"].iloc[:-1].to_numpy(dtype=float)  # (Ta - Ti)
    q_solar   = df_week["solarradiation"].iloc[:-1].to_numpy(dtype=float)

    if "is_night" in df_week.columns:
        idx_night = df_week["is_night"].iloc[:-1].fillna(0).astype(int).to_numpy()
    else:
        idx_night = np.zeros(L, dtype=int)

    # measured dTi/dt (length L)
    delta_T_i = (df_week["Internal_Air_Temperature"].diff().iloc[1:].to_numpy(dtype=float) / delta_t_hours)

    # exact q for plotting (length L)
    q_real = df_q_week_aligned["Q_spc_exact"].iloc[:-1].to_numpy(dtype=float)

    # --- 2R term (length L) ---
    if use_2R:
        T_i_k = df_week["Internal_Air_Temperature"].iloc[:-1].to_numpy(dtype=float)
        T_m_k = df_week["T_m_24h"].iloc[:-1].to_numpy(dtype=float)
        delta_T_m = (T_m_k - T_i_k)
    else:
        delta_T_m = 0.0  # scalar broadcasts safely

    # ---- implied q_hat_sim (length L) ----
    # q_hat = C*dT - dt*(deltaTa/Ra) - dt*(deltaTm/Rm) - w_s*q_s - w*dt - w_n*idx
    q_hat_sim = (delta_T_i * C
                 - (delta_T_a / R_a) * delta_t_hours
                 - (delta_T_m / R_m) * delta_t_hours if use_2R else 0.0)

    # careful: previous line becomes ambiguous with ternary; do explicit:
    if use_2R:
        q_hat_sim = (delta_T_i * C
                     - (delta_T_a / R_a) * delta_t_hours
                     - (delta_T_m / R_m) * delta_t_hours
                     - w_s * q_solar
                     - w_const * delta_t_hours
                     - w_n * idx_night)
    else:
        q_hat_sim = (delta_T_i * C
                     - (delta_T_a / R_a) * delta_t_hours
                     - w_s * q_solar
                     - w_const * delta_t_hours
                     - w_n * idx_night)

    # simulate Ti with q_real
    q_opt = q_real.copy()

    time_idx_state = df_week.index[1:]  # length L
    Ti_meas_state  = df_week["Internal_Air_Temperature"].iloc[1:].to_numpy(dtype=float)

    if use_2R:
        # ---------- 2R1C: Euler ----------
        delta_T_i_sim = (
            (delta_T_a / R_a) * delta_t_hours
            + (delta_T_m / R_m) * delta_t_hours
            + q_opt
            + w_s * q_solar
            + w_const * delta_t_hours
            + w_n * idx_night
        ) / C
    else:
        # ---------- 1R1C: exact discretization ----------
        a = np.exp(-delta_t_hours / (R_a * C))

        T_i_k = df_week["Internal_Air_Temperature"].iloc[:-1].to_numpy(dtype=float)
        T_a_k = df_week["temp"].iloc[:-1].to_numpy(dtype=float)

        forcing = (
            q_opt
            + w_s * q_solar
            + w_n * idx_night
            + T_a_k / R_a
        )

        # exact discrete update: Ti(k+1) - Ti(k)
        delta_T_i_sim = (
            a * T_i_k
            + (1 - a) * R_a * forcing
            - T_i_k
        ) / delta_t_hours


    Ti_sim_free   = np.zeros(L, dtype=float)
    Ti_sim_reinit = np.zeros(L, dtype=float)

    Ti_prev_free   = float(df_week["Internal_Air_Temperature"].iloc[0])
    Ti_prev_reinit = float(df_week["Internal_Air_Temperature"].iloc[0])

    for k, t in enumerate(time_idx_state):
        Ti_sim_free[k] = Ti_prev_free + delta_T_i_sim[k] * delta_t_hours
        Ti_prev_free = Ti_sim_free[k]

        if (t.hour == 0) and (t.minute == 0):
            if t in df_week.index:
                Ti_prev_reinit = float(df_week.loc[t, "Internal_Air_Temperature"])

        Ti_sim_reinit[k] = Ti_prev_reinit + delta_T_i_sim[k] * delta_t_hours
        Ti_prev_reinit = Ti_sim_reinit[k]

    rmse_q_week   = rmse(q_hat_sim, q_real)
    rmse_T_free   = rmse(Ti_sim_free, Ti_meas_state)
    rmse_T_reinit = rmse(Ti_sim_reinit, Ti_meas_state)

    return {
        "L": L,
        "time_q": df_week.index[:-1],
        "time_state": time_idx_state,
        "q_real": q_real,
        "q_hat_sim": q_hat_sim,
        "Ti_meas": Ti_meas_state,
        "Ti_sim_free": Ti_sim_free,
        "Ti_sim_reinit": Ti_sim_reinit,
        "rmse_q_week": rmse_q_week,
        "rmse_T_free": rmse_T_free,
        "rmse_T_reinit": rmse_T_reinit,
    }


In [65]:
# %%
def run_allhomes_validation(df_params, df_detached, df_q_exact_all):
    rows = []

    for _, prow in df_params.iterrows():
        id_use = prow[id_col]
        train_end = prow[train_end_col]

        # Some IDs might be numeric vs string across datasets; normalize by trying both
        # (most of your data uses df_detached["Property_ID"] exactly, so keep as-is)
        try:
            df_home = build_df_home(df_detached, id_use)
        except Exception:
            # try casting
            try:
                df_home = build_df_home(df_detached, str(id_use))
                id_use = str(id_use)
            except Exception:
                continue

        try:
            week_start, week_end, df_week, df_q_aligned = find_validation_month(
                id_use=id_use,
                df_home=df_home,
                df_q_exact_all=df_q_exact_all,
                train_end=train_end,
            )
        except Exception:
            continue

        try:
            sim = simulate_week(prow, df_week, df_q_aligned, delta_t_hours=DELTA_T_HOURS)
        except Exception:
            continue

        rows.append({
            "Property_ID": id_use,
            "train_end": train_end,
            "val_week_start": week_start,
            "val_week_end": week_end,
            "rmse_q_week": sim["rmse_q_week"],
            "rmse_T_free": sim["rmse_T_free"],
            "rmse_T_reinit": sim["rmse_T_reinit"],
            "n_points": len(df_week),
        })

    return pd.DataFrame(rows)

In [66]:
# %%
def run_validation_given_windows(df_params, windows):
    rows = []
    for _, prow in df_params.iterrows():
        id_use = str(prow["Property_ID"])
        if id_use not in windows:
            continue

        start, end = windows[id_use]

        try:
            df_home = build_df_home(df_detached, id_use)
            df_month = df_home.loc[start:end]

            df_q = read_q_exact_30min(df_q_exact_all, id_use, start, end)
            df_q = df_q.reindex(df_month.index)

            sim = simulate_week(prow, df_month, df_q, DELTA_T_HOURS)
        except Exception:
            continue

        out = {
            "Property_ID": id_use,
            "rmse_q": sim["rmse_q_week"],
            "rmse_T_free": sim["rmse_T_free"],
            "rmse_T_reinit": sim["rmse_T_reinit"],
            "n_points": len(df_month),
            "C": float(prow.get("C", np.nan)),
            "R_a": float(prow.get("R_a", np.nan)),
            "w_s": float(prow.get("w_s", np.nan)),
            "w_n": float(prow.get("w_n", np.nan)),
            "train_start": prow.get("train_start", pd.NaT),
            "train_end": prow.get("train_end", pd.NaT),
        }
        if "R_m" in prow.index:
            out["R_m"] = float(prow.get("R_m", np.nan))
        rows.append(out)

    return pd.DataFrame(rows)



In [67]:
def get_latest_train_end(id_use):
    ends = []
    for dfp in methods.values():
        row = dfp[dfp["Property_ID"] == id_use].iloc[0]
        ends.append(pd.Timestamp(row["train_end"]))
    return max(ends)


In [ ]:
# ============================
# DROP-IN: put this AFTER all function definitions
# ============================

# ---- Load data ----
df_detached = pd.read_parquet(DETACHED_PARQUET)
df_q_exact_all = pd.read_parquet(Q_STREAMS_PARQUET)

# Ensure datetime index for df_detached subset operations
if "Timestamp" in df_detached.columns:
    df_detached["Timestamp"] = pd.to_datetime(df_detached["Timestamp"])
    df_detached = df_detached.set_index("Timestamp")
df_detached = df_detached.sort_index()

# ---- Methods (your 4 only) ----
methods = {
    "Linear regression (aggregated) 1R": df_params_agg_1R,
    "Linear regression (disaggregated) 1R": df_params_dis_1R,
    "Linear regression (aggregated) 2R": df_params_agg_2R,
    "Linear regression (disaggregated) 2R": df_params_dis_2R,
}

# Ensure consistent ID + datetime types
for dfp in methods.values():
    dfp["Property_ID"] = dfp["Property_ID"].astype(str)
    dfp["train_end"] = pd.to_datetime(dfp["train_end"], errors="coerce")

# ---- Intersection of homes across all 4 methods ----
common_ids = set.intersection(*(set(dfp["Property_ID"]) for dfp in methods.values()))
print("Homes in all 4 methods:", len(common_ids))

# ---- Helper: latest train_end across the 4 methods for a given home ----
def get_latest_train_end_local(id_use: str) -> pd.Timestamp:
    ends = []
    for dfp in methods.values():
        r = dfp.loc[dfp["Property_ID"] == id_use, "train_end"]
        if r.empty or pd.isna(r.iloc[0]):
            raise ValueError(f"Missing train_end for {id_use} in one method.")
        ends.append(pd.Timestamp(r.iloc[0]))
    return max(ends)

# ---- Build shared validation windows (post latest-train-end) ----
val_windows = {}
for id_use in sorted(common_ids):
    id_use = str(id_use)
    try:
        latest_train_end = get_latest_train_end_local(id_use)
        df_home = build_df_home(df_detached, id_use)

        start, end, _, _ = find_validation_month(
            id_use=id_use,
            df_home=df_home,
            df_q_exact_all=df_q_exact_all,
            train_end=latest_train_end,
        )
        val_windows[id_use] = (start, end)

    except Exception as e:
        # Drop homes that can't support a shared window
        print(f"Drop {id_use}: {e}")

print("Final homes with shared validation windows:", len(val_windows))

# ---- Validate each method on ONLY homes in its dataset AND in val_windows ----
val_frames = []
for label, dfp in methods.items():
    keep_ids = set(dfp["Property_ID"]).intersection(val_windows.keys())
    dfp_sub = dfp[dfp["Property_ID"].isin(keep_ids)].copy()

    dfv = run_validation_given_windows(dfp_sub, val_windows)
    dfv["method"] = label
    val_frames.append(dfv)

df_val_all = pd.concat(val_frames, ignore_index=True)

# ---- Summaries ----
print(df_val_all.groupby("method")[["rmse_T_reinit", "rmse_q"]].describe())

# ---- Box plots (same analysis) ----
plt.figure(figsize=(9, 4))
df_val_all.boxplot(column="rmse_T_reinit", by="method", grid=True, rot=20)
plt.suptitle("")
plt.title("Validation RMSE of Indoor Temperature (daily reinit)")
plt.ylabel("RMSE (°C)")
plt.tight_layout()
plt.ylim(0, 5)
plt.show()

# Optional: q RMSE box plot (keeps same style)
plt.figure(figsize=(9, 4))
df_val_all.boxplot(column="rmse_q", by="method", grid=True, rot=20)
plt.suptitle("")
plt.title("Validation RMSE of Q (implied vs exact)")
plt.ylabel("RMSE (kWh)")
plt.tight_layout()
plt.show()

# Optional: R_m distribution (only meaningful for 2R methods)
if "R_m" in df_val_all.columns and df_val_all["R_m"].notna().any():
    plt.figure(figsize=(9, 4))
    df_val_all.boxplot(column="R_m", by="method", grid=True, rot=20)
    plt.suptitle("")
    plt.title("Estimated R_m by method (2R only where available)")
    plt.ylabel("R_m")
    plt.tight_layout()
    plt.show()


Homes in all 4 methods: 186


In [ ]:
# ============================
# DROP-IN: Example plots for ONLY the 4 LR methods you’re comparing
# (put after the validation / boxplot block)
# ============================

def plot_example_week_all_methods_4LR(id_use, days=7):
    id_use = str(id_use)
    if id_use not in val_windows:
        raise KeyError(f"{id_use} not in val_windows")

    # shared window (month) -> take first N days for plotting
    start, end = val_windows[id_use]

    df_home = build_df_home(df_detached, id_use)
    df_week = df_home.loc[start:start + pd.Timedelta(days=days)].copy()

    df_q = read_q_exact_30min(df_q_exact_all, id_use, df_week.index[0], df_week.index[-1])
    df_q = df_q.reindex(df_week.index)
    if df_q["Q_spc_exact"].isna().any():
        raise RuntimeError("NaNs in Q_spc_exact after reindex for example plot window.")

    # run sims for your 4 methods (same window)
    sims = {}
    for label, dfp in methods.items():
        prow = dfp.loc[dfp["Property_ID"] == id_use].iloc[0]
        sims[label] = simulate_week(prow, df_week, df_q, DELTA_T_HOURS)

    # --------- Temperature plot (reinit) ----------
    # Use any sim for time_state / Ti_meas (they share df_week)
    any_key = next(iter(sims.keys()))

    plt.figure(figsize=(13, 4))
    plt.plot(
        sims[any_key]["time_state"],
        sims[any_key]["Ti_meas"],
        color="k", linewidth=2, label="Measured"
    )

    for k, s in sims.items():
        plt.plot(
            s["time_state"],
            s["Ti_sim_reinit"],
            label=f"{k} (RMSE={s['rmse_T_reinit']:.2f})"
        )

    plt.title(f"Example validation {days}-day window – Home {id_use}")
    plt.ylabel("Indoor temperature (°C)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # --------- Q plot (real vs implied) ----------
    plt.figure(figsize=(13, 4))
    plt.plot(
        sims[any_key]["time_q"],
        sims[any_key]["q_real"],
        color="k", linewidth=2, label="Q_spc_exact"
    )

    for k, s in sims.items():
        plt.plot(
            s["time_q"],
            s["q_hat_sim"],
            label=f"{k} (RMSE_q={s['rmse_q_week']:.2f})"
        )

    plt.title(f"Example validation {days}-day window – Home {id_use}")
    plt.ylabel("Q (kWh)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ---- Pick a representative home from the evaluated set ----
# (guaranteed to exist in val_windows + to have validation results)
example_id = df_val_all["Property_ID"].dropna().astype(str).iloc[8]
plot_example_week_all_methods_4LR(example_id, days=7)

